# EGFR Atomistic Frustration Pipeline
**Chen et al. (2020) Nature Communications** — Independent Reimplementation  
25 EGFR–inhibitor complexes (expanded from paper's 4)

---
**Run order:** Stage 0 → 1 → 2 (unit tests) → 3 (validation) → 4 (EGFR analysis)

## Stage 0 — Environment Setup & PyRosetta Verification

In [ ]:
import sys, os
os.chdir("/home/tugba/egfr_atomic_resolution")
sys.path.insert(0, "src")

# Verify all dependencies
import numpy as np
import pandas as pd
import scipy
import matplotlib
import yaml
import requests
from pathlib import Path
print(f"numpy {np.__version__}, pandas {pd.__version__}, scipy {scipy.__version__}")

from openbabel import openbabel as ob
print(f"OpenBabel {ob.OBReleaseVersion()}")


In [ ]:
import pyrosetta
pyrosetta.init("-mute all")
print("PyRosetta OK")

# Quick test: score lysozyme
import requests
pdb_path = Path("data/raw_pdb/1LYZ.pdb")
if not pdb_path.exists():
    r = requests.get("https://files.rcsb.org/download/1LYZ.pdb", timeout=30)
    pdb_path.parent.mkdir(parents=True, exist_ok=True)
    pdb_path.write_bytes(r.content)

pose = pyrosetta.pose_from_pdb(str(pdb_path))
sfxn = pyrosetta.create_score_function("ref2015")
score = sfxn(pose)
print(f"1LYZ: {pose.total_residue()} residues, REF2015 score = {score:.2f} REU")
print("Stage 0 PASSED ✓")


## Stage 1 — Data Preparation

Download all 25 PDB structures, extract ligands, generate Rosetta .params files.

In [ ]:
import yaml
cfg = yaml.safe_load(open("config.yaml"))

# Show candidate list
df_cands = pd.read_csv(cfg["paths"]["candidates_csv"])
print(f"Candidates: {len(df_cands)} structures")
print(df_cands[["pdb_id","ligand_id","affinity_type","affinity_pM","resolution_A"]].to_string(index=False))


In [ ]:
from prepare_structures import process_structure

results = []
for _, row in df_cands.iterrows():
    res = process_structure(row["pdb_id"], row["ligand_id"], cfg)
    results.append(res)

df_prep = pd.DataFrame(results)
ok = df_prep["status"].isin(["ok","skipped"]).sum()
print(f"\nPreparation: {ok}/{len(df_prep)} structures successful")
print(df_prep[["pdb_id","status"]].to_string(index=False))


In [ ]:
# Verify poses load correctly
from run_pipeline import load_pose_with_ligand, find_ligand_resnum

summaries = []
sfxn = pyrosetta.create_score_function("ref2015")

for _, row in df_cands.iterrows():
    pid, lid = row["pdb_id"], row["ligand_id"]
    pdb_f   = Path(cfg["paths"]["processed"]) / f"{pid}_clean.pdb"
    par_f   = Path(cfg["paths"]["params"])    / f"{pid}_{lid}.params"
    if not pdb_f.exists() or not par_f.exists():
        summaries.append({"pdb_id": pid, "status": "missing"})
        continue
    try:
        pose  = load_pose_with_ligand(str(pdb_f), str(par_f))
        score = sfxn(pose)
        lig_r = find_ligand_resnum(pose, lid)
        summaries.append({
            "pdb_id": pid,
            "n_residues": pose.total_residue(),
            "lig_resnum": lig_r,
            "score_REU": round(score, 2),
            "status": "ok",
        })
        print(f"  {pid}: {pose.total_residue()} res, lig@{lig_r}, score={score:.1f}")
    except Exception as e:
        summaries.append({"pdb_id": pid, "status": f"error: {e}"})
        print(f"  {pid} ERROR: {e}")

df_poses = pd.DataFrame(summaries)
print(f"\nSuccessfully loaded: {(df_poses['status']=='ok').sum()}/{len(df_poses)}")
print("Stage 1 PASSED ✓")


## Stage 2 — Unit Tests

Verify Eq. 1, Eq. 2 implementation on a small protein before running full analysis.

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, "-m", "pytest", "src/test_frustration.py", "-v", "--tb=short"],
    capture_output=True, text=True, cwd="/home/tugba/egfr_atomic_resolution"
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])


In [ ]:
# Quick manual sanity check: pairwise energy symmetry
from frustration import get_protein_contacts, pairwise_energy, native_aa_frequency

pose_test = pyrosetta.pose_from_pdb("data/raw_pdb/1LYZ.pdb")
sfxn(pose_test)
e_59 = pairwise_energy(pose_test, sfxn, 5, 9, exclude_fa_rep=True)
e_95 = pairwise_energy(pose_test, sfxn, 9, 5, exclude_fa_rep=True)
print(f"e(5,9) = {e_59:.6f}, e(9,5) = {e_95:.6f}")
print(f"Symmetry OK: {abs(e_59 - e_95) < 1e-6}")

aa_freq = native_aa_frequency(pose_test)
print(f"\nAmino acid frequency (sum={sum(aa_freq.values()):.6f}):")
print({k: f"{v:.3f}" for k, v in sorted(aa_freq.items())})
print("Stage 2 PASSED ✓")


## Stage 3 — Validation on Lysozyme (1LYZ)

Expected: buried core contacts → mostly minimally frustrated  
Surface contacts → more neutral/highly frustrated

In [ ]:
from run_pipeline import run_validation
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

df_lyz = run_validation(cfg, n_decoys=cfg["frustration"]["n_decoys"])

print(f"\nLysozyme frustration results (n_decoys={cfg['frustration']['n_decoys']}):")
print(df_lyz["frustration_class"].value_counts())
print(f"\nFrustration index stats:")
print(df_lyz["F_index"].describe().round(3))


In [ ]:
# Display validation figure
img = mpimg.imread("results/validation_lysozyme.png")
fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(img)
ax.axis("off")
plt.tight_layout()
plt.show()

# Core/surface check via SASA
from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley
import numpy as np

parser = PDBParser(QUIET=True)
struct = parser.get_structure("lyz", "data/raw_pdb/1LYZ.pdb")
sr = ShrakeRupley()
sr.compute(struct, level="R")

sasa_by_seqid = {}
for res in struct.get_residues():
    sasa_by_seqid[res.id[1]] = res.sasa

# Map contacts to core/surface
SASA_BURIED_THRESH = 20  # Å² — considered buried if < threshold
core_contacts = []
surf_contacts = []
for _, row in df_lyz.iterrows():
    s1 = sasa_by_seqid.get(row["resi"], 999)
    s2 = sasa_by_seqid.get(row["resj"], 999)
    avg_sasa = (s1 + s2) / 2
    if avg_sasa < SASA_BURIED_THRESH:
        core_contacts.append(row["frustration_class"])
    else:
        surf_contacts.append(row["frustration_class"])

core_min_frac = core_contacts.count("minimally_frustrated") / len(core_contacts) if core_contacts else 0
surf_min_frac = surf_contacts.count("minimally_frustrated") / len(surf_contacts) if surf_contacts else 0

print(f"Core contacts (SASA < {SASA_BURIED_THRESH} Å²): {len(core_contacts)} total")
print(f"  → {100*core_min_frac:.0f}% minimally frustrated")
print(f"Surface contacts: {len(surf_contacts)} total")
print(f"  → {100*surf_min_frac:.0f}% minimally frustrated")
print(f"\n✓ Core > Surface: {core_min_frac > surf_min_frac}")
print("Stage 3 PASSED ✓" if core_min_frac > surf_min_frac else "Stage 3 WARNING: unexpected pattern, review code")


## Stage 4 — EGFR Analysis

Run frustration survey on all 25 EGFR–inhibitor complexes and correlate
minimally frustrated ligand-pocket contacts with binding affinity.

In [ ]:
# --- 4a. Quick prototype with n_decoys=50 ---
# Increase n_decoys in config.yaml for final analysis (200-1000)
from run_pipeline import run_all_egfr
import yaml

cfg["frustration"]["n_decoys"] = 50  # override for prototype
df_results = run_all_egfr(cfg, n_decoys=50)
print(df_results[["pdb_id","ligand_id","affinity_pM","n_minimally_frustrated","n_contacts_total"]])


In [ ]:
# --- 4b. Display correlation figure ---
img = mpimg.imread("results/egfr_correlation.png")
fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(img)
ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# --- 4c. Sub-analysis: Kd-only structures ---
from scipy import stats
import numpy as np

df_kd = df_results[df_results["affinity_type"].str.upper().isin(["KD","KI"])].copy()
print(f"Kd/Ki-only structures: {len(df_kd)}")

if len(df_kd) >= 3:
    x = df_kd["n_minimally_frustrated"].values
    y = df_kd["log10_affinity_pM"].values
    r, p = stats.pearsonr(x, y)
    print(f"Kd-only Pearson r = {r:.3f}, p = {p:.3f}")

# Full set
if len(df_results) >= 3:
    r_all, p_all = stats.pearsonr(
        df_results["n_minimally_frustrated"].values,
        df_results["log10_affinity_pM"].values
    )
    print(f"Full set Pearson  r = {r_all:.3f}, p = {p_all:.3f} (n={len(df_results)})")

print("\nStage 4 COMPLETE ✓")


In [ ]:
# --- 4d. Save results README ---
readme = f"""# EGFR Frustration Analysis Results

## Parameters
- n_decoys: {cfg['frustration']['n_decoys']}
- protein-protein contact cutoff: {cfg['contacts']['protein_protein_cutoff_A']} Å
- ligand-protein contact cutoff:  {cfg['contacts']['ligand_protein_cutoff_A']} Å
- seed: {cfg['frustration']['seed']}
- exclude_fa_rep: {cfg['frustration']['exclude_fa_rep']}
- chain used for multi-chain structures: {cfg['chain_selection']['default']}

## Affinity data
- 8 Kd structures (thermodynamic) + 17 IC50 structures
- Covalent inhibitors excluded: 4LQM (DJK), 5XDK (8JC), 3IKA (0UN)

## Key files
- egfr_frustration_summary.csv  — per-structure results
- egfr_correlation.png          — scatter plot (all 25)
- validation_lysozyme.png       — Stage 3 validation
"""
with open("results/README.md", "w") as f:
    f.write(readme)
print("results/README.md written")
